In [1]:
# Creates or appends one log record to: <log_folder>/<file_name>.log.json
# or: <log_folder>/<file_name>.log.csv
#
# Returns the path to the log file.    
# =========================================================================

import uuid
import csv
import json
from pathlib import Path
from typing import Literal

FIELD_NAMES = [
    "log_uid",
    "source_name",
    "file_name",
    "source_url",
    "lakehouse_path",        
    "bronze_table_name",
    "load_type",
    "ingestion_started_at_utc",
    "ingestion_completed_at_utc",
    "download_time_seconds",
    "write_time_seconds",
    "size_bytes",
    "rows",
    "status",
    "error_message"
]

def write_log(        
        source_name,
        file_name,
        source_url,
        lakehouse_path,        
        bronze_table_name,
        load_type,
        ingestion_started_at_utc,
        ingestion_completed_at_utc,
        download_time_seconds,
        write_time_seconds,
        size_bytes,
        rows,
        status,
        error_message=None,
        log_format: Literal["json", "csv"] = "json",
        log_folder="/lakehouse/default/Files/logs"
    ) -> str:         

    if log_format not in {"json", "csv"}:
        raise ValueError("log_format must be either 'json' or 'csv'")

    log_record = {
        "log_uid": str(uuid.uuid4()),
        "source_name": source_name,
        "file_name": file_name,
        "source_url": source_url,
        "lakehouse_path": lakehouse_path,        
        "bronze_table_name": bronze_table_name,
        "load_type": load_type,
        "ingestion_started_at_utc": ingestion_started_at_utc,
        "ingestion_completed_at_utc": ingestion_completed_at_utc,
        "download_time_seconds": download_time_seconds,
        "write_time_seconds": write_time_seconds,
        "size_bytes": size_bytes,
        "rows": rows,
        "status": status,
        "error_message": error_message
    }

    log_directory = Path(log_folder)
    log_directory.mkdir(parents=True, exist_ok=True)

    # Prevent a path contained in file_name from affecting log_folder.
    safe_file_name = Path(file_name).name
    log_path = log_directory / f"{safe_file_name}.log.{log_format}"

    if log_format == "json":
        # One JSON object per line, making the file safely appendable.
        with log_path.open("a", encoding="utf-8") as log_file:
            json.dump(log_record, log_file, ensure_ascii=False, default=str)
            log_file.write("\n")

    else:
        file_already_exists = log_path.exists()

        with log_path.open("a", encoding="utf-8", newline="") as log_file:
            writer = csv.DictWriter(
                log_file,
                fieldnames=FIELD_NAMES,
                delimiter=","
            )

            if not file_already_exists:
                writer.writeheader()

            writer.writerow(log_record)

    return str(log_path)    

StatementMeta(, 8aa72b09-3486-4338-8a64-bbd9bf832830, 3, Finished, Available, Finished, False)